In [1]:
import warnings
warnings.simplefilter(action='ignore')

import numpy as np
import pandas as pd

import pickle
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import train_test_split
import os
import matplotlib.pyplot as plt

import matplotlib
from astromcad.astromcad import *

2024-08-08 18:06:17.733731: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-08-08 18:06:18.462452: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9373] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-08 18:06:18.463901: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-08 18:06:18.631932: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1534] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-08-08 18:06:18.954902: I tensorflow/core/platform/cpu_feature_guar

In [2]:
file_names_load = ['lc_classnum_Ia-norm_90.pickle',
 'lc_classnum_Ia-91bg_67.pickle',
 'lc_classnum_Iax_52.pickle',
'lc_classnum_Ib.pickle',
 'lc_classnum_Ic.pickle',
 'lc_classnum_Ic-BL.pickle',
 'lc_classnum_II.pickle',
 'lc_classnum_IIn.pickle',
 'lc_classnum_IIb.pickle',
 'lc_classnum_TDE_64.pickle',
 'lc_classnum_SLSN-I_60.pickle',
 'lc_classnum_AGN_70_old.pickle',
 'lc_classnum_CART_63_old.pickle',
 'lc_classnum_Kilonova_99.pickle',
'lc_classnum_PISN_61_old.pickle',
'lc_classnum_ILOT_62_old.pickle',
'lc_classnum_uLens-BSR_90_old.pickle']

file_names = ['lc_classnum_Ia.pickle',
 'lc_classnum_Ia-91bg.pickle',
 'lc_classnum_Iax.pickle',
'lc_classnum_Ib.pickle',
 'lc_classnum_Ic.pickle',
 'lc_classnum_Ic-BL.pickle',
 'lc_classnum_II.pickle',
 'lc_classnum_IIn.pickle',
 'lc_classnum_IIb.pickle',
 'lc_classnum_TDE.pickle',
 'lc_classnum_SLSN-I.pickle',
 'lc_classnum_AGN_old.pickle',
 'lc_classnum_CART_old.pickle',
 'lc_classnum_Kilonova.pickle',
'lc_classnum_PISN_old.pickle',
'lc_classnum_ILOT_old.pickle',
'lc_classnum_uLens-BSR.pickle']

classes = ['SNIa', 'SNIa-91bg', 'SNIax', 'SNIb', 'SNIc', 'SNIc-BL', 'SNII', 'SNIIn', 'SNIIb', 'TDE', 'SLSN-I', 'AGN', 'CaRT', 'KNe', 'PISN', 'ILOT', 'uLens-BSR']

file_to_class = dict(zip(file_names, classes))


class_to_file = {v: k for k, v in file_to_class.items()}
class_to_file['SNIa-x'] = class_to_file['SNIax']
class_to_file['SNIa-norm'] = class_to_file['SNIa']

anom_classes = file_names[-5:]
non_anom_classes = file_names[:-5]


color = ['r', 'g', 'y', 'b', 'purple', 'orange', 'gray', 'k', 'm', 'c', 'brown', 'olive']
colors = color

font = {'size'   : 17}

matplotlib.rc('font', **font)
anom_classes = [file_to_class[i] for i in anom_classes]
non_anom_classes = [file_to_class[i] for i in non_anom_classes]


In [3]:
def save(save_path , obj):
    with open(save_path, 'wb') as f:
        pickle.dump(obj, f)

def load(file_name):
    with open(file_name, 'rb') as f:
        return pickle.load(f)

target = load("data/target")
x_data = load("data/x_data")
host_galaxy_info = load("data/host_galaxy_info")


In [4]:
valid = [False] * len(target)

for class_ in np.unique(target):
    cnt = 0
    for i in range(len(target)):

        if (target[i] == class_):
            valid[i] = True
            cnt+=1
        if (cnt == 13000): # cut down
            break
for i in range(len(target) - 1, -1, -1):
    if not valid[i]:
        del target[i]
        del x_data[i]
        del host_galaxy_info[i]

lengths = []

for lc in x_data:
    lengths.append(len(lc))

ntimesteps = np.max(lengths)

print(ntimesteps)

# Pad for TF masking layer and time dilation
def dilate(vals, red):
    return ((((vals * 100) - 30) / (1 + red)) + 30) / 100 # unscale time then correct


for ind in range(len(x_data)):
  x_data[ind][:, 1] = dilate(x_data[ind][:, 1], host_galaxy_info[ind][0])
  x_data[ind] = np.pad(x_data[ind], ((0, ntimesteps - len(x_data[ind])), (0, 0)))
# x_data = np.array(x_data)
# Split data

y_data_anom = []
y_data_norm = []
x_data_norm = []
x_data_anom = []
host_gal_anom = []
host_gal = []

for i in range(len(target)):

    if (file_to_class[target[i]] in anom_classes):
        x_data_anom.append(x_data[i])
        y_data_anom.append(target[i])
        host_gal_anom.append(host_galaxy_info[i])

    else:
        x_data_norm.append(x_data[i])
        y_data_norm.append(target[i])
        host_gal.append(host_galaxy_info[i])

656


In [5]:

# One-hot Encoding
print(np.unique(y_data_norm))
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder(handle_unknown='ignore')

y_data_norm = enc.fit_transform(np.array(y_data_norm).reshape(-1, 1)).todense()
# Train-test split

X_train, X_test, host_gal_train, host_gal_test, y_train, y_test = train_test_split(x_data_norm, host_gal, y_data_norm, random_state = 40, test_size = 0.1)
X_train, X_val, host_gal_train, host_gal_val, y_train, y_val = train_test_split(X_train, host_gal_train, y_train, random_state = 40, test_size = 1/9)
class_weights = {i : 0 for i in range(y_train.shape[1])}

for value in y_train:
  class_weights[np.argmax(value)]+=1

for id in class_weights.keys():
  class_weights[id] = len(y_train) / class_weights[id]

X_train = np.array(X_train)
y_train = np.array(y_train)

X_val = np.array(X_val)
y_val = np.array(y_val)

X_test = np.array(X_test)
y_test = np.array(y_test)

host_gal_train = np.array(host_gal_train)
host_gal_test = np.array(host_gal_test)
host_gal_val = np.array(host_gal_val)

y_train = np.squeeze(y_train)
y_val = np.squeeze(y_val)
y_test = np.squeeze(y_test)
y_data_anom = np.squeeze(y_data_anom)
X_train.shape

['lc_classnum_AGN_old.pickle' 'lc_classnum_II.pickle'
 'lc_classnum_IIb.pickle' 'lc_classnum_IIn.pickle'
 'lc_classnum_Ia-91bg.pickle' 'lc_classnum_Ia.pickle'
 'lc_classnum_Iax.pickle' 'lc_classnum_Ib.pickle'
 'lc_classnum_Ic-BL.pickle' 'lc_classnum_Ic.pickle'
 'lc_classnum_SLSN-I.pickle' 'lc_classnum_TDE.pickle']


(95182, 656, 4)

In [6]:
dummy = enc.transform(np.array([class_to_file[i] for i in non_anom_classes]).reshape(-1, 1))

ordered_class_names = [-1 for i in range(len(non_anom_classes))]

for ind, i in enumerate(dummy.todense()):
    ordered_class_names[np.argmax(i)] = non_anom_classes[ind]

In [7]:
ordered_class_names

['AGN',
 'SNII',
 'SNIIb',
 'SNIIn',
 'SNIa-91bg',
 'SNIa',
 'SNIax',
 'SNIb',
 'SNIc-BL',
 'SNIc',
 'SLSN-I',
 'TDE']

In [8]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, TimeDistributed, Dense, Masking, concatenate, GRU, LSTM
from tensorflow.keras.callbacks import EarlyStopping
from math import dist
import keras
def make_classifier_model(latent_size):

    num_classes = len(class_weights)
    n_features = 4

    input_1 = Input((ntimesteps, n_features), name='lc')  # X.shape = (Nobjects, Ntimesteps, 4) CHANGE

    masking_input1 = Masking(mask_value=0.)(input_1)

    lstm1 = GRU(100, return_sequences=True, activation='tanh')(masking_input1)
    lstm2 = GRU(100, return_sequences=False, activation='tanh')(lstm1)

    dense1 = Dense(100, activation='tanh')(lstm2)

    input_2 = Input(shape = (2, ), name='host') # CHANGE

    dense2 = Dense(10)(input_2)

    merge1 = concatenate([dense1, dense2])

    dense3 = Dense(100, activation='relu')(merge1)

    dense4 = Dense(latent_size, activation='relu', name='latent')(dense3)

    output = Dense(num_classes, activation='softmax')(dense4)

    model = keras.Model(inputs=[input_1, input_2], outputs=output)

    model.compile(loss = "categorical_crossentropy", optimizer="adam", metrics=['accuracy'])
    
    return model

def make_classifier_no_host(latent_size):

    num_classes = len(class_weights)
    n_features = 4

    input_1 = Input((ntimesteps, n_features), name='lc')  # X.shape = (Nobjects, Ntimesteps, 4) CHANGE

    masking_input1 = Masking(mask_value=0.)(input_1)

    lstm1 = GRU(100, return_sequences=True, activation='tanh')(masking_input1)
    lstm2 = GRU(100, return_sequences=False, activation='tanh')(lstm1)

    dense1 = Dense(100, activation='tanh')(lstm2)

#     input_2 = Input(shape = (2, ), name='host') # CHANGE

#     dense2 = Dense(10)(input_2)

#     merge1 = concatenate([dense1, dense2])

#     dense3 = Dense(100, activation='relu')(merge1)

    dense4 = Dense(latent_size, activation='relu', name='latent')(dense1)

    output = Dense(num_classes, activation='softmax')(dense4)

    model = keras.Model(inputs=[input_1], outputs=output)

    model.compile(loss = "categorical_crossentropy", optimizer="adam", metrics=['accuracy'])
    
    return model

In [9]:
# Measure feature correlation of the latent space

def train_model(latent_size, host=True):
    if (host):
        model = make_classifier_model(latent_size) # HERE YOU CAN CHANGE THE LATENT SPACE SIZE
    else:
        model = make_classifier_no_host(latent_size) # HERE YOU CAN CHANGE THE LATENT SPACE SIZE
        
    early_stopping = EarlyStopping(
                                  patience=5,
                                  min_delta=0.001,                               
                                  monitor="val_loss",
                                  restore_best_weights=True
                                  )



    try:
        if (host):
            history = model.fit(x = [X_train, host_gal_train], validation_data=([X_val, host_gal_val], y_val), y = y_train, epochs=40, batch_size = 128, class_weight = class_weights, callbacks=[early_stopping])
        else:
            history = model.fit(x = [X_train], validation_data=([X_val], y_val), y = y_train, epochs=40, batch_size = 128, class_weight = class_weights, callbacks=[early_stopping])
            
    except Exception as e:
        print(e)
        pass
    
    
    return model


In [11]:

model = train_model(100, False)

2024-08-03 15:09:23.363904: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1926] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31127 MB memory:  -> device: 0, name: Tesla V100-SXM2-32GB, pci bus id: 0000:3a:00.0, compute capability: 7.0


Epoch 1/40


2024-08-03 15:09:33.077861: W tensorflow/core/common_runtime/type_inference.cc:339] Type inference failed. This indicates an invalid graph that escaped type checking. Error message: INVALID_ARGUMENT: expected compatible input types, but input 1:
type_id: TFT_OPTIONAL
args {
  type_id: TFT_PRODUCT
  args {
    type_id: TFT_TENSOR
    args {
      type_id: TFT_INT32
    }
  }
}
 is neither a subtype nor a supertype of the combined inputs preceding it:
type_id: TFT_OPTIONAL
args {
  type_id: TFT_PRODUCT
  args {
    type_id: TFT_TENSOR
    args {
      type_id: TFT_INT8
    }
  }
}

	for Tuple type infernce function 0
	while inferring type of node 'cond_35/output/_22'
2024-08-03 15:09:34.053439: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:467] Loaded cuDNN version 90000
2024-08-03 15:09:36.573739: I external/local_xla/xla/service/service.cc:168] XLA service 0x1531c0766690 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-08-03 15:09

744/744 [==============================] - 28s 25ms/step - loss: 22.3139 - accuracy: 0.3669 - val_loss: 1.3993 - val_accuracy: 0.5137
Epoch 2/40
744/744 [==============================] - 16s 22ms/step - loss: 16.0272 - accuracy: 0.5608 - val_loss: 1.0368 - val_accuracy: 0.6284
Epoch 3/40
744/744 [==============================] - 16s 22ms/step - loss: 13.5474 - accuracy: 0.6318 - val_loss: 0.8881 - val_accuracy: 0.6773
Epoch 4/40
744/744 [==============================] - 16s 22ms/step - loss: 12.1387 - accuracy: 0.6733 - val_loss: 0.8126 - val_accuracy: 0.7085
Epoch 5/40
744/744 [==============================] - 16s 22ms/step - loss: 11.0960 - accuracy: 0.7054 - val_loss: 0.7354 - val_accuracy: 0.7406
Epoch 6/40
744/744 [==============================] - 16s 22ms/step - loss: 10.4147 - accuracy: 0.7269 - val_loss: 0.6841 - val_accuracy: 0.7601
Epoch 7/40
744/744 [==============================] - 16s 22ms/step - loss: 9.6417 - accuracy: 0.7499 - val_loss: 0.6575 - val_accuracy: 0.76

In [12]:
model.save("models/FinalModel.keras")